# 2. Datos: ETL y EDA

Este capítulo reúne el ETL y el análisis exploratorio de los entregables anteriores (secciones 2.1 a 2.8, que se
muestran ejecutadas con sus figuras e interpretación) y resume los hallazgos que guiaron el diseño del modelo original.
El EDA interactivo está en la página **EDA** del dashboard.

In [1]:
# Localiza la raíz del proyecto (la carpeta con src/ y app.py) para poder importar los módulos de src/
import sys
from pathlib import Path
raiz = Path.cwd().resolve()
while not ((raiz / "src").is_dir() and (raiz / "app.py").exists()):
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
# Librerías de análisis y módulos propios: configuración, datos, EDA, evaluación y utilidades del libro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from src import config as cfg, datos, eda, evaluacion as ev
from src.libro import md, tabla, mostrar_plotly
from src.eda import es, p_fmt
# Estilo común de los gráficos, semilla global (SEMILLA = 42) y formato de las tablas
eda.configurar_graficos()
cfg.fijar_semillas()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## Fuente y ETL

- **Fuente:** catálogo sísmico de la Red Sismológica Nacional de Colombia, Servicio Geológico Colombiano (SGC),
  portales histórico y actual (10 archivos), 2004-01 a 2026-09.
- **Transformaciones:** unificación de esquemas, eliminación de duplicados entre portales, depuración de coordenadas y
  profundidades imposibles, filtro de magnitud M ≥ 2,5 (completitud del catálogo) y asignación a celdas de 0,5°.
- **Carga:** `data/sismos_sgc_2004_2026.csv` (catálogo limpio) y el panel celda × mes que construye `src/panel.py`.

In [2]:
from src import data as D
cat, P = D.catalogo(), D.panel()
obs = P[P.y.notna()]
tabla(pd.DataFrame([
    ("Sismos (M ≥ 2,5)", f"{len(cat):,}".replace(",", " ")),
    ("Periodo", f"{cat.time.min():%Y-%m} a {cat.time.max():%Y-%m}"),
    ("Celdas activas (≥ 20 sismos antes de 2021)", P.celda.nunique()),
    ("Filas celda × mes con objetivo", f"{len(obs):,}".replace(",", " ")),
    ("Variables predictoras (benchmarks)", 18),
    ("Variables adicionales del modelo original", "13 de excitación + 12 de historia"),
    ("Celdas-mes sin sismos", f"{100 * (obs.y == 0).mean():.1f} %"),
], columns=["concepto", "valor"]))

,concepto,valor
0,"Sismos (M ≥ 2,5)",40 365
1,Periodo,2004-01 a 2026-09
2,Celdas activas (≥ 20 sismos antes de 2021),163
3,Filas celda × mes con objetivo,42 380
4,Variables predictoras (benchmarks),18
5,Variables adicionales del modelo original,13 de excitación + 12 de historia
6,Celdas-mes sin sismos,77.1 %


## Hallazgos del EDA que guían el modelado

1. **Exceso de ceros y sobredispersión.** La mayoría de las celdas-mes no tiene sismos y la varianza supera con creces a
   la media: motiva la pérdida binomial negativa inflada en ceros del modelo B y la métrica RMSLE.
2. **Persistencia temporal.** La actividad de un mes anticipa la del siguiente (ACF alta en rezagos cortos): motiva las
   variables de rezago, la persistencia como miembro y la excitación tipo ETAS.
3. **Dependencia espacial.** Moran y Knox muestran agrupamiento y difusión entre celdas vecinas: motiva las variables de
   vecindad y la excitación espacial.
4. **Heterogeneidad entre celdas y deriva.** Celdas muy activas (nido de Bucaramanga) conviven con celdas casi
   inactivas, y hay cambios de régimen: motiva un combinador cuyos pesos dependen del contexto de cada celda (A) y el
   reentrenamiento anual.
5. **Sin fuga.** La auditoría de fuga (sección 2.8) confirma que ninguna variable usa información del mes objetivo.